# Day 18 ｜ SQLite：把排盘记录存成能查的东西

今天不是学 SQL 语法全集，是学**四件事**：
1. 表结构怎么定（DDL 是设计，不是副产品）
2. `SELECT / INSERT / UPDATE / DELETE` 四个动作，以及 **UPDATE / DELETE 忘写 WHERE 会怎样**
3. 数据库里的「空」有三种，只有一种是对的
4. 为什么聚合要下推给 SQL，而不是捞进 Python 再循环

数据源是你自己的项目：`paipan/data/charts.db`，由 `paipan/db.py` 建表、`api.py` 写入。
**库文件不进 git**（`.gitignore` 里 `*.db`、`data/` 两条都挡住了），所以它可以随便折腾。

先在终端建一次表（跑这个 notebook 前）：

```
.venv/bin/python paipan/db.py
```

看到 `插入 3 行，自增 id = [1, 2, 3]` 就够了——它会重置数据，方便你反复练。

## 第 1 步 ｜ 先看清这个库里到底有什么

SQLite 把自己的结构存在一张叫 `sqlite_master` 的系统表里。**先查它，别猜。**

**先猜**：这个库里现在有几个对象？分别是什么？

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), Path.cwd().parent,
                        Path.home() / "Documents/AI自学/ai-product-journey"]
            if (p / "paipan" / "data" / "charts.db").exists())
sys.path.insert(0, str(ROOT / "paipan"))
import sqlite3
from db import connect, init

con = connect()
print("库文件:", ROOT / "paipan/data/charts.db")
for r in con.execute("SELECT type, name FROM sqlite_master ORDER BY type, name"):
    print(" ", r["type"], "→", r["name"])
print()
print("行数:", con.execute("SELECT COUNT(*) FROM charts").fetchone()[0])

一个表 + 一个索引。索引是 `db.py` 的 DDL 里那句
`CREATE INDEX idx_charts_name ON charts(name, created_at)` —— 它为什么建在这两列上，第 6 步揭晓。

### 表结构本身也能查：PRAGMA

**先猜**：`name` 这一列允许 NULL 吗？它的默认值是什么？

In [ ]:
for r in con.execute("PRAGMA table_info(charts)"):
    d = dict(r)
    print(f"  {d['name']:12s} {d['type']:8s} 非空={d['notnull']} 默认值={d['dflt_value']} 主键={d['pk']}")

注意两个细节，它们是 SQLite 的脾气，不是 bug：

- **类型是宽松的**。SQLite 的 TEXT 列你塞数字它也收（叫 type affinity）。所以「字段类型」在 SQLite 里是**约定**，不是强制——真正拦得住你的只有 `NOT NULL` 和 `CHECK`。
- **`birth_time` 的 `notnull=0`**，也就是它可以是 NULL。这是故意的：**时辰未知是一个真实业务状态**，Day 17 接口里那条 `time_known` 就是为它服务的。

## 第 2 步 ｜ SELECT：查，而且只查你要的

三个层次，从粗到细。

In [ ]:
print("① 全捞（练手可以，生产别写）")
for r in con.execute("SELECT * FROM charts"):
    print("  ", dict(r)["name"], dict(r)["birth_date"], dict(r)["day_pillar"])

In [ ]:
print("② 指定列 + 条件 + 排序 + 限量")
sql = """SELECT name, birth_date, birth_time, day_pillar, sect, created_at
         FROM charts
         WHERE gender = '男'
         ORDER BY created_at DESC, id DESC
         LIMIT 5"""
for r in con.execute(sql):
    print("  ", r["name"], r["birth_date"], str(r["birth_time"]), "日柱", r["day_pillar"], "sect", r["sect"])
print()
print("③ 只取一个数：COUNT")
print("   男性行数:", con.execute("SELECT COUNT(*) FROM charts WHERE gender='男'").fetchone()[0])

## 关键：口径（sect）为什么必须存在每一行里

看这两行——同一天、同一个人，日柱不同：

In [ ]:
print("同一个生日的两种情况：")
for r in con.execute("""SELECT birth_date, birth_time, day_pillar, time_known, sect
                        FROM charts WHERE birth_date = '2000-10-15' ORDER BY id"""):
    print(f"   {r['birth_date']} 时辰={str(r['birth_time']):>6} → 日柱 {r['day_pillar']}"
          f"｜time_known={r['time_known']}｜sect={r['sect']}")
print()
print("如果表里没存 sect：三个月后你改了口径，这批老记录到底是哪个口径算的？没人知道。")
print("存了 sect，任何一行都能被重新验算 —— 这叫数据的可复现性，不是洁癖。")

### 补一课：排序到底按 `id` 还是 `created_at`（我这段改过两遍，理由也错过一次）

我原先写的是 `ORDER BY created_at DESC, id DESC`，给的理由是「同一秒不稳，加唯一列兜底」。
那句话**只能解释第二列为什么存在，解释不了第一列凭什么存在**。跑一遍就知道差在哪。

In [ ]:
# 造一张「插入顺序 ≠ 业务时间」的表 —— 回填历史记录时就是这个局面
con.execute("DELETE FROM charts")
for name, ts in [("甲", "2000-01-01 10:00:00"),
                 ("乙", "2000-01-01 10:00:00"),      # 故意和甲同一秒
                 ("丙", "2000-01-01 09:00:00")]:     # 业务上更早，但最后一个插入
    con.execute("""INSERT INTO charts (name,gender,birth_date,year_pillar,month_pillar,
                   day_pillar,day_master,sect,created_at)
                   VALUES (?,'男','2000-01-01','x','y','z','w',1,?)""", (name, ts))
con.commit()
print("表（按 id 看写入顺序）：")
for r in con.execute("SELECT id, name, created_at FROM charts ORDER BY id"):
    print(f"   id={r['id']} {r['name']} {r['created_at']}")

for label, sql in [
    ("id DESC", "SELECT name FROM charts ORDER BY id DESC"),
    ("created_at DESC", "SELECT name FROM charts ORDER BY created_at DESC"),
    ("created_at DESC, id DESC", "SELECT name FROM charts ORDER BY created_at DESC, id DESC"),
]:
    order = " → ".join(r["name"] for r in con.execute(sql))
    plan = [r["detail"] for r in con.execute("EXPLAIN QUERY PLAN " + sql)]
    cost = "要建 TEMP B-TREE（全表排序）" if any("TEMP" in p for p in plan) else "倒走主键，不排序"
    print(f"  {label:26s} {order:14s} ｜ {cost}")

三个结论，第 1 条就是对你那个问题的直接回答：

1. **`ORDER BY id DESC` 单独用就够了，而且更好。** `id` 是主键、唯一 → 顺序 100% 确定；
   执行计划显示它**不需要建临时排序树**，倒着走主键就是结果。你昨天问「为什么不直接 id DESC」——
   正确答案是：现在这张表就该用它。
2. **两列写法的唯一正当理由是「第一列不唯一」。** 看第 2、3 行结果完全一样，就是证据：
   第二列只在打平时参与，平时它一点作用都没有，却照样要为第一列的排序付 TEMP B-TREE 的代价。
3. **什么时候才真需要 `created_at` 打头？** 当「业务时间」和「写入顺序」分家时——
   回填历史数据、或者以后加了 `updated_at`。上面那张表里丙 09:00 却最后一个插入，
   两种写法就给出了两个答案。现在我们的 `created_at` 永远等于插入时刻，所以两者重合，
   既然重合，就选更便宜、更确定的那个。

⚠️ 顺带纠正我另一句过头话：我原稿写「同一秒你每次看到的顺序可能不同」。实测 5 次，
同一秒的甲乙**每次都是同一顺序**。准确的说法是——

> **观察到稳定 ≠ SQL 语义保证稳定。**

排序列不唯一时，SQL 标准允许数据库任意决定平局次序；换版本、换查询计划、加索引都可能变。
兜底列买的是「保证」，不是「我刚才看的一样」。用观察冒充保证，是 Day 15 #22 那个病的变体。

⏸ **暂停点 1**：到这一步，「库里有什么」「怎么查」都过关了。

## 第 3 步 ｜ 三种「空」，只有一种是对的

⚠️ **这一格会先 `DELETE FROM charts` 清空整张表**，用两条自己造的行来演示「空」的三种写法。
练完想恢复成 3 条样例，回终端再跑一次 `.venv/bin/python paipan/db.py` 就行
（它每次都从空表开始，所以是可复现的）。

这是今天最容易踩的坑，而且它**不报错**。

| 你写的 | 意思 | 结果 |
|---|---|---|
| `WHERE col IS NULL` | 真的不知道 | ✅ 能查到 |
| `WHERE col = ''` | 知道，且是空字符串 | ⚠️ 查的是另一种东西 |
| `WHERE col = NULL` | **永远不成立** | ❌ 0 行，而且不报错 |

SQL 里 `NULL = NULL` 不是真，是 `NULL`（未知）。所以 `= NULL` 永远筛不出东西。

In [ ]:
con.execute("DELETE FROM charts")
con.execute("""INSERT INTO charts (name,gender,birth_date,birth_time,year_pillar,month_pillar,
                                 day_pillar,day_master,sect)
               VALUES ('甲某','男','2000-01-01',NULL,'x','y','z','w',1)""")
con.execute("""INSERT INTO charts (name,gender,birth_date,birth_time,year_pillar,month_pillar,
                                 day_pillar,day_master,sect)
               VALUES ('乙某','男','2000-01-02','','x','y','z','w',1)""")
con.commit()

tests = [
    ("IS NULL     （真的不知道）", "SELECT COUNT(*) n FROM charts WHERE birth_time IS NULL"),
    ("= ''        （知道，值为空串）", "SELECT COUNT(*) n FROM charts WHERE birth_time = ''"),
    ("= NULL      （永远不成立）", "SELECT COUNT(*) n FROM charts WHERE birth_time = NULL"),
    ("IS NOT NULL （包含空串！）", "SELECT COUNT(*) n FROM charts WHERE birth_time IS NOT NULL"),
]
for label, sql in tests:
    print(f"  {label:26s} → {con.execute(sql).fetchone()['n']} 行")
print()
print("最后一行是关键：空字符串也算\'已知\'，所以它满足 IS NOT NULL。")
print("如果业务里用空串表示\'无\'，IS NOT NULL 就把\'无\'和\'有\'混成一堆——")
print("这就是 Day 16 那条：空值只有一个含义 = 不知道；\'不是库\'这种事实要写个明确的值。")

### 约束：数据库当场拒绝脏值

`gender` 列上有 `CHECK (gender IN ('男','女'))`。试试写进去会怎样。

**先猜**：写 `'中'` 会被拒绝，那写空串 `''` 呢？

In [ ]:
for bad in ["中", "", "男"]:
    try:
        con.execute("INSERT INTO charts (name,gender,birth_date,year_pillar,month_pillar,"
                    "day_pillar,day_master,sect) VALUES ('测试',?,'2000-01-03','x','y','z','w',1)",
                    (bad,))
        con.commit()
        print(f"  gender={bad!r:6} ✅ 写进去了")
    except sqlite3.IntegrityError as e:
        con.rollback()
        print(f"  gender={bad!r:6} ❌ 被拦：{e}")

## 第 4 步 ｜ 你亲手写：INSERT / UPDATE / DELETE

这一节是今天的主体。**规矩：先 SELECT 看有几行受影响，再执行。**
因为 `UPDATE` 和 `DELETE` 不写 `WHERE` 就是全表操作，而**它不会问你确认**。

### ① 亲手插一条

把下面三个空填上（名字随意、性别只能男/女、日期用 YYYY-MM-DD）：

In [ ]:
# ↓↓ 这三个默认值是能跑通的示例，把它们改成你自己的（改完重跑这一格）↓↓
my_name = "练手"        # 比如 "张三"
my_gender = "男"        # 只能是 男 或 女
my_date = "1995-08-08"  # YYYY-MM-DD

cur = con.execute("""INSERT INTO charts (name, gender, birth_date, birth_time, time_known,
                           year_pillar, month_pillar, day_pillar, hour_pillar, day_master, sect)
       VALUES (?,?,?,?,?,?,?,?,?,?,?)""",
    (my_name, my_gender, my_date, None, 0, "甲子", "乙丑", "丙寅", None, "丙", 1))
con.commit()
print("插入成功，自增 id =", cur.lastrowid)
print("查回来看看：", dict(con.execute("SELECT * FROM charts WHERE id = ?", (cur.lastrowid,)).fetchone()))

注意我传的是 `None` 而不是 `''`——Python 的 `None` 会变成 SQL 的 NULL，字符串空值会存成 `''`。
**这是你在 Python 和 SQL 之间翻译「没有」这个概念的唯一机会**，想清楚再传。

### ② 亲手改一条

先数受影响几行，再改。这就是「先 SELECT 后 UPDATE」的纪律。

In [ ]:
target = my_name
print("改之前命中几行:", con.execute("SELECT COUNT(*) FROM charts WHERE name = ?", (target,)).fetchone()[0])

# ↓↓ 你写 WHERE 条件：只改 id 最大的那条（别改成全表） ↓↓
cur = con.execute("UPDATE charts SET name = ? WHERE id = (SELECT MAX(id) FROM charts)",
                  (target + "_已改名",))
con.commit()
print("实际改了", cur.rowcount, "行")
print("验证:", [r["name"] for r in con.execute("SELECT name FROM charts ORDER BY id")])

### ③ 亲手删一条

`deleted` 应该是 1。如果你删完发现是 3，说明你的 WHERE 写宽了。

In [ ]:
rows_before = con.execute("SELECT COUNT(*) FROM charts").fetchone()[0]
print("删之前总行数:", rows_before)

# ↓↓ 你写：删掉刚才改名那条 ↓↓
cur = con.execute("DELETE FROM charts WHERE name = ?", ("{}_已改名".format(my_name),))
con.commit()
print("deleted =", cur.rowcount, "｜总行数从", rows_before, "变成",
      con.execute("SELECT COUNT(*) FROM charts").fetchone()[0])
print()
print("如果上面 rowcount 是 0，你的 WHERE 没命中——这就是先 SELECT 的意义。")

## 第 5 步 ｜ 聚合下推：别把 CSV 的思路带进数据库

**先猜**：要统计「每个日主各有多少人」，捞全表进 Python 用 `Counter` 数，和在 SQL 里 `GROUP BY`，差别在哪？

In [ ]:
from collections import Counter

print("① Python 侧数（把全表读进内存）")
cnt = Counter(r["day_master"] for r in con.execute("SELECT day_master FROM charts"))
print("  ", dict(cnt))

print("② SQL 侧数（聚合下推）")
for r in con.execute("""SELECT day_master, COUNT(*) AS n FROM charts
                        GROUP BY day_master ORDER BY n DESC, day_master"""):
    print(f"   {r['day_master']} {r['n']}")
print()
print("两条独立路径，结果必须一致 —— Day 15 的交叉验证在 SQL 里同样好用。")
print("差别在规模：几百行无所谓，几百万行时 ① 会把你内存吃掉，而且慢得多。")

## 第 6 步 ｜ 索引：用 EXPLAIN QUERY PLAN 看它到底有没有走

第 1 步那个 `idx_charts_name` 建在 `(name, created_at)` 上。为什么是这两列？
因为你的产品最主要的查询就是「按人查历史，新的在前」。验证一下。

In [ ]:
for label, sql in [
    ("WHERE name = ?          （有索引）", "SELECT * FROM charts WHERE name = '甲某'"),
    ("WHERE birth_date = ?    （没索引）", "SELECT * FROM charts WHERE birth_date = '2000-01-01'"),
]:
    plan = [r["detail"] for r in con.execute("EXPLAIN QUERY PLAN " + sql)]
    print(f"  {label}")
    print(f"     {plan}")
print()
print("SEARCH USING INDEX = 走索引直接定位；SCAN = 全表扫描，一行一行看。")
print("你现在只有几行，两者都快——但这句话在几百万行时会变成你产品的第一号性能问题。")

## 第 7 步 ｜ SQL 注入：为什么必须用 `?`

**先猜**：`"SELECT * FROM charts WHERE name = '" + 用户输入 + "'"`，用户在名字里填 `'; DROP TABLE charts; --` 会怎样？

In [ ]:
evil = "'; DROP TABLE charts; --"

# ❌ 错误写法（别在项目里这样，这里只是演示它有多脆弱）
naive = "SELECT * FROM charts WHERE name = '" + evil + "'"
print("拼出来的 SQL：", naive)
print()
# ✅ 正确写法：参数化，? 占位
rows = con.execute("SELECT * FROM charts WHERE name = ?", (evil,)).fetchall()
print("参数化查询命中行数:", len(rows))
print("表还在吗:", con.execute("SELECT COUNT(*) FROM charts").fetchone()[0], "行")
print()
print("关键理解：参数化不是「过滤掉危险字符」，而是把数据永远当数据——")
print("它不可能被解释成 SQL 语句。过滤是补救，参数化是免疫。")

## 收尾

**1. `NOTES.md` 三行**（Day 18）。

**2. 提交**（存盘 ≠ 提交）：

```
cd ~/Documents/AI自学/ai-product-journey
git add paipan/db.py paipan/api.py .gitignore notebooks/18_sqlite.ipynb NOTES.md
git status
```

**看一眼 `git status` 里有没有 `charts.db`**——不该有，`.gitignore` 的 `*.db` 和 `data/` 双重挡住。确认没有再：

```
git commit -m "Day18：排盘记录落 SQLite，接口加存取/查历史/统计/删除"
git push origin main
```

提交信息照你实际做的改一遍。

## 三个带走的判断

1. **DDL 是设计**：先写下 `SCHEMA` 那段，再写任何一行查询。表结构定错，代码全要返工。
2. **`UPDATE` / `DELETE` 永远先 `SELECT COUNT(*)` 用同一个 WHERE 数一遍**。数据库不会拦你。
3. **CSV 和数据库不是同一张表**：批量样本（只读、整块生成、拿来验算）vs 用户记录（一条条插、要按人查、要改要删）。混成一张，两边都做不好。

## 加餐（选做）

- 给表加一列 `note TEXT`（用户备注），想想已有的库文件怎么加列（提示：`ALTER TABLE ... ADD COLUMN`）
- 用 `EXPLAIN QUERY PLAN` 查一下「按日主统计」走不走索引，然后试着建一个，看计划变不变
- 想一个产品问题：同一个人重复排盘，是每次都存一行（历史），还是覆盖？各有什么代价？